# TODO 6.2：AWQ 质量、回归和性能评估

本 Notebook 使用已经导出的 AWQ g32/g64/g128 模型，复用项目现有 FinQA 指标和 Benchmark。完整模型从 `banking_llm_project/awq` 读取，评测结果写入 `OptimizationSystem/out/awq`。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
QUALITY_CONFIG = PROJECT_DIR / 'configs/baseline_quality.json'
BENCHMARK_CONFIG = PROJECT_DIR / 'configs/benchmark.json'
AWQ_RESULT_DIR = PROJECT_DIR / 'out/awq'
BASELINE_PREDICTIONS = PROJECT_DIR / 'out/baseline_quality_experiment/baseline_predictions.jsonl'
sys.path.insert(0, str(PROJECT_DIR))
from src.data.finqa_assets import load_assets
from src.evaluation.baseline_quality import run_baseline_quality, validate_quality_report
from src.evaluation.finqa_metrics import classify_numeric_regression
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.quantization.awq import load_awq_service
assets = load_assets(DATASET_DIR)
quality_config = json.loads(QUALITY_CONFIG.read_text(encoding='utf-8'))
benchmark_config = json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
print('质量集:', len(assets.quality_dev), '回归集:', len(assets.high_risk_regression))

In [ ]:
baseline_rows = [json.loads(line) for line in BASELINE_PREDICTIONS.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(baseline_rows) == len(assets.quality_dev), 'FP16 基线预测数量不完整。'
baseline_by_question = {row['question']: row for row in baseline_rows}
print('FP16 基线预测：PASS')

In [ ]:
from src.baseline.service import BaselineService, InferenceRequest
baseline_service = BaselineService.from_local_model(
    config_file=BASELINE_CONFIG,
    model_manifest_file=PROJECT_DIR / 'out/model_input_real/model_manifest.json',
    log_file=AWQ_RESULT_DIR / 'fp16_regression_service.jsonl',
)
def predict_rows(service, dataset, prefix):
    rows = []
    for start in range(0, len(dataset), int(quality_config['batch_size'])):
        stop = min(start + int(quality_config['batch_size']), len(dataset))
        batch = [dataset[index] for index in range(start, stop)]
        requests = [InferenceRequest.from_finqa_row(row, request_id=f'{prefix}-{start + offset + 1:06d}') for offset, row in enumerate(batch)]
        for row, request, result in zip(batch, requests, service.generate_batch(requests, prompt_suffix=quality_config.get('prompt_suffix', ''))):
            rows.append({'question': request.question, 'prediction': result['output_text'], 'reference': row.get('answer') or row.get('qa', {}).get('answer')})
    return rows
baseline_regression_rows = predict_rows(baseline_service, assets.high_risk_regression, 'fp16-regression')
print('FP16 高风险回归基线：', len(baseline_regression_rows))

In [ ]:
for plan_id in ('awq-int4-w4a16-g32', 'awq-int4-w4a16-g64', 'awq-int4-w4a16-g128'):
    plan_dir = AWQ_RESULT_DIR / plan_id
    manifest_file = plan_dir / 'quantized_model_manifest.json'
    quality_dir = plan_dir / 'quality'
    service = load_awq_service(
        baseline_config_file=BASELINE_CONFIG,
        quantized_manifest_file=manifest_file,
        source_model_manifest_file=MODEL_MANIFEST,
        log_file=plan_dir / 'awq_quality_service.jsonl',
    )
    quality_report = run_baseline_quality(
        service=service, dataset=assets.quality_dev,
        dataset_manifest=assets.manifest, quality_config=quality_config,
        output_dir=quality_dir,
    )
    quality_report_path = quality_dir / 'baseline_quality_report.json'
    quality_report_path.write_text(json.dumps(quality_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    quality_validation = validate_quality_report(quality_report_path)
    (quality_dir / 'validation.json').write_text(json.dumps(quality_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    predictions = [json.loads(line) for line in (quality_dir / 'baseline_predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    regressions = []
    for row in predictions:
        baseline = baseline_by_question.get(row['question'])
        if baseline is None:
            continue
        regressions.append({**classify_numeric_regression(baseline['prediction'], row['prediction'], row['reference'], float(quality_config.get('tolerance', 1e-4))), 'question': row['question']})
    regression_payload = {
        'schema_version': 1, 'plan_id': plan_id, 'dataset': quality_report['dataset'],
        'total': len(regressions), 'new_regressions': sum(row['new_regression'] for row in regressions),
        'critical_errors': sum(row['critical_error_type'] is not None for row in regressions),
        'rows': regressions,
    }
    (plan_dir / 'regression.json').write_text(json.dumps(regression_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(plan_id, quality_report['metrics'], 'new_regressions=', regression_payload['new_regressions'])

In [ ]:
for plan_id in ('awq-int4-w4a16-g32', 'awq-int4-w4a16-g64', 'awq-int4-w4a16-g128'):
    plan_dir = AWQ_RESULT_DIR / plan_id
    candidate_service = load_awq_service(BASELINE_CONFIG, plan_dir / 'quantized_model_manifest.json', MODEL_MANIFEST, plan_dir / 'awq_regression_service.jsonl')
    candidate_rows = predict_rows(candidate_service, assets.high_risk_regression, f'{plan_id}-regression')
    baseline_map = {row['question']: row for row in baseline_regression_rows}
    comparisons = []
    for row in candidate_rows:
        baseline_row = baseline_map[row['question']]
        comparisons.append({**classify_numeric_regression(baseline_row['prediction'], row['prediction'], row['reference'], float(quality_config.get('tolerance', 1e-4))), 'question': row['question']})
    regression_payload = {'schema_version': 1, 'plan_id': plan_id, 'dataset_role': 'high_risk_regression', 'total': len(comparisons), 'new_regressions': sum(row['new_regression'] for row in comparisons), 'critical_errors': sum(row['critical_error_type'] is not None for row in comparisons), 'rows': comparisons}
    (plan_dir / 'regression.json').write_text(json.dumps(regression_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(plan_id, 'high-risk regressions=', regression_payload['new_regressions'])

In [ ]:
for plan_id in ('awq-int4-w4a16-g32', 'awq-int4-w4a16-g64', 'awq-int4-w4a16-g128'):
    plan_dir = AWQ_RESULT_DIR / plan_id
    manifest_file = plan_dir / 'quantized_model_manifest.json'
    service = load_awq_service(BASELINE_CONFIG, manifest_file, MODEL_MANIFEST, plan_dir / 'awq_benchmark_service.jsonl')
    benchmark_report = run_benchmark(service, assets.calibration_prompts[0], benchmark_config)
    report_file = plan_dir / 'benchmark_report.json'
    report_file.write_text(json.dumps(benchmark_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    benchmark_validation = validate_benchmark_report(report_file)
    (plan_dir / 'benchmark_validation.json').write_text(json.dumps(benchmark_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(plan_id, 'benchmark:', benchmark_validation.complete)

完成后请将每个 plan 目录中的 `quality/`、`regression.json`、`benchmark_report.json` 和两个 JSONL 服务日志保留在 `out/awq`，模型权重仍只保留在 `banking_llm_project/awq`。